U2 AquaVigia: modelo de series de tiempo (criterio 7)

Dimension U2 de Patricia Apaza: temperatura y oxigeno disuelto en tiempo real.

Se entrena con el HISTORIAL REAL del USGS un modelo que proyecta cada variable a UNA HORA (12 pasos de 5 min), se valida contra dos referencias (repetir el ultimo valor y predecir la media) y se guarda el ganador para aplicarlo despues en streaming (cuaderno C).
Tambien se calculan los umbrales de advertencia por estacion y la incertidumbre del modelo, que sirven para estimar el riesgo de alerta.

In [1]:
import os, json, time, math
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

INTEGRANTE = "Patricia Apaza"
VARIABLES = ["temperatura_c","oxigeno_mgl"]

BASE = "/opt/s12-u2-usgs"
ARTIFACTS = f"{BASE}/artifacts"
DATA = f"{BASE}/data/historial.jsonl"
RUTA_MODELOS = f"{ARTIFACTS}/modelos_patricia"
os.makedirs(RUTA_MODELOS, exist_ok=True)
if not os.path.exists(DATA):
    raise FileNotFoundError(f"Falta {DATA}. Copia historial.jsonl a la carpeta data de esta sesion.")

spark = (SparkSession.builder.appName("u2-usgs-modelo").master("local[*]")
         .config("spark.sql.shuffle.partitions", "8").config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

esquema_evento = (StructType().add("evento_id", StringType()).add("estacion_id", StringType())
    .add("variable", StringType()).add("codigo_parametro", StringType()).add("valor", DoubleType())
    .add("fecha_medicion_utc", StringType()).add("fecha_ingesta_utc", StringType())
    .add("retraso_min", DoubleType()).add("origen", StringType()).add("fuente", StringType()))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 06:00:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Historial real: que hay

Lecturas del USGS por estacion y variable. Solo se usan las variables de esta dimension.

In [2]:
h = (spark.read.schema(esquema_evento).json(DATA)
     .where(F.col("variable").isin(VARIABLES) & F.col("valor").isNotNull())
     .withColumn("ts", F.to_timestamp("fecha_medicion_utc")))
print("lecturas del historial (mis variables):", h.count())
resumen = (h.groupBy("estacion_id", "variable").agg(F.count("*").alias("lecturas"), F.min("ts").alias("desde"),
           F.max("ts").alias("hasta"), F.round(F.avg("valor"), 2).alias("promedio")))
resumen.orderBy("estacion_id", "variable").show(60, truncate=False)

lecturas del historial (mis variables): 646003


[Stage 3:==============>                                          (4 + 12) / 16]

+---------------+-------------+--------+-------------------+-------------------+--------+
|estacion_id    |variable     |lecturas|desde              |hasta              |promedio|
+---------------+-------------+--------+-------------------+-------------------+--------+
|USGS-01491000  |oxigeno_mgl  |25886   |2026-08-01 00:40:00|2026-10-29 23:30:00|6.83    |
|USGS-01491000  |temperatura_c|25898   |2026-08-01 00:40:00|2026-10-30 00:30:00|21.4    |
|USGS-01579550  |oxigeno_mgl  |25880   |2026-08-01 00:40:00|2026-10-30 00:05:00|8.59    |
|USGS-01579550  |temperatura_c|25880   |2026-08-01 00:40:00|2026-10-30 00:05:00|24.3    |
|USGS-01589290  |temperatura_c|25623   |2026-08-01 00:40:00|2026-10-29 23:50:00|19.7    |
|USGS-01594441  |oxigeno_mgl  |25864   |2026-08-01 00:40:00|2026-10-30 00:05:00|7.64    |
|USGS-01594441  |temperatura_c|25864   |2026-08-01 00:40:00|2026-10-30 00:05:00|21.67   |
|USGS-01640152  |oxigeno_mgl  |25878   |2026-08-01 00:40:00|2026-10-30 00:05:00|8.32    |
|USGS-0164

## 2. Serie regular de 5 minutos y tabla ancha

Se redondea cada lectura a su bloque de 5 minutos y se deja una columna por variable. Solo se conservan estaciones que tienen TODAS las variables de la dimension.

In [3]:
PASO = 300
base = (h.withColumn("ts5", (F.floor(F.col("ts").cast("long") / PASO) * PASO).cast("long"))
        .groupBy("estacion_id", "variable", "ts5").agg(F.avg("valor").alias("valor")))
ancho = (base.groupBy("estacion_id", "ts5").pivot("variable", VARIABLES).agg(F.first("valor"))
         .na.drop(subset=VARIABLES).cache())
print("filas (estacion x 5 min con todas mis variables):", ancho.count())
print("estaciones utilizables:", ancho.select("estacion_id").distinct().count())
ancho.orderBy("estacion_id", "ts5").show(5)

filas (estacion x 5 min con todas mis variables): 309899
estaciones utilizables: 12
+-------------+----------+-------------+-----------+
|  estacion_id|       ts5|temperatura_c|oxigeno_mgl|
+-------------+----------+-------------+-----------+
|USGS-01491000|1785562800|         25.3|        5.5|
|USGS-01491000|1785563100|         25.3|        5.5|
|USGS-01491000|1785563400|         25.2|        5.5|
|USGS-01491000|1785563700|         25.2|        5.5|
|USGS-01491000|1785564000|         25.2|        5.5|
+-------------+----------+-------------+-----------+
only showing top 5 rows


## 3. Variables del modelo

Para proyectar una variable `v` una hora adelante se usan: su valor actual, sus rezagos (5, 10, 15, 30 y 60 min), el cambio de la ultima hora, el valor actual de las otras variables de la dimension y la hora del dia (seno y coseno).
Solo se admiten filas con los rezagos y la meta EXACTAMENTE a los tiempos esperados (sin huecos).

In [4]:
LAGS = [1, 2, 3, 6, 12]
HORIZONTE = 12      # 12 pasos de 5 min = 1 hora

def columnas(v):
    return [v] + [f"{v}_l{k}" for k in LAGS] + ["delta_1h"] + [u for u in VARIABLES if u != v] + ["hora_sin", "hora_cos"]

def construir(v):
    w = Window.partitionBy("estacion_id").orderBy("ts5")
    d = (ancho.withColumn("ts5_meta", F.lead("ts5", HORIZONTE).over(w))
              .withColumn("label", F.lead(v, HORIZONTE).over(w)))
    cond = (F.col("ts5_meta") - F.col("ts5") == HORIZONTE * PASO)
    for k in LAGS:
        d = d.withColumn(f"{v}_l{k}", F.lag(v, k).over(w)).withColumn(f"_t{k}", F.lag("ts5", k).over(w))
        cond = cond & (F.col("ts5") - F.col(f"_t{k}") == k * PASO)
    d = (d.where(cond)
          .withColumn("delta_1h", F.col(v) - F.col(f"{v}_l12"))
          .withColumn("hora", (F.col("ts5") % 86400) / 3600.0)
          .withColumn("hora_sin", F.sin(F.col("hora") * (2 * math.pi / 24)))
          .withColumn("hora_cos", F.cos(F.col("hora") * (2 * math.pi / 24))))
    return d.select("estacion_id", "ts5", "label", *columnas(v)).na.drop()

CORTE = ancho.approxQuantile("ts5", [0.8], 0.001)[0]
print("Corte cronologico (80 % del tiempo):", pd.to_datetime(CORTE, unit="s", utc=True))

Corte cronologico (80 % del tiempo): 2026-10-12 03:35:00+00:00


## 4. Entrenamiento, validacion y eleccion del ganador

Division cronologica (el entrenamiento solo usa lo que termina antes del corte; la prueba, lo posterior). Se comparan dos modelos contra dos referencias con RMSE, MAE y R2.

In [5]:
ev = {m: RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}
def evaluar(df, nombre):
    r = {"Modelo": nombre, "RMSE": ev["rmse"].evaluate(df), "MAE": ev["mae"].evaluate(df), "R2": ev["r2"].evaluate(df)}
    print(f"  {nombre}: RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  R2={r['R2']:.4f}")
    return r

TABLAS, META = {}, {"variables": {}, "corte": int(CORTE), "horizonte_min": HORIZONTE * 5, "integrante": INTEGRANTE}
for v in VARIABLES:
    print(f"\n===== {v} =====")
    ds = construir(v).cache()
    train = ds.where(F.col("ts5") + HORIZONTE * PASO <= CORTE)
    test = ds.where(F.col("ts5") > CORTE)
    n_tr, n_te = train.count(), test.count()
    print(f"  filas entrenamiento={n_tr}  prueba={n_te}")
    if n_tr < 500 or n_te < 100:
        print("  POCOS DATOS: baja la exigencia o baja mas historial (--dias 90)."); continue
    cols = columnas(v)
    media = train.agg(F.avg("label")).first()[0]
    filas = [evaluar(test.withColumn("prediction", F.col(v)), "Referencia: repetir el ultimo valor"),
             evaluar(test.withColumn("prediction", F.lit(media)), "Referencia: media de entrenamiento")]
    modelos = {}
    ens = lambda: VectorAssembler(inputCols=cols, outputCol="features", handleInvalid="skip")
    cand = {"LinearRegression": LinearRegression(featuresCol="features", labelCol="label", regParam=0.01),
            "RandomForest": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=40, maxDepth=7, seed=42)}
    for nombre, est in cand.items():
        m = Pipeline(stages=[ens(), est]).fit(train)
        modelos[nombre] = m
        filas.append(evaluar(m.transform(test), nombre))
    tabla = pd.DataFrame(filas)
    ganador = tabla[tabla["Modelo"].isin(cand)].sort_values("RMSE").iloc[0]
    mejor = modelos[ganador["Modelo"]]
    mejor.write().overwrite().save(f"{RUTA_MODELOS}/modelo_{v}_1h")
    rm_p = tabla.loc[0, "RMSE"]; rm_m = tabla.loc[1, "RMSE"]
    META["variables"][v] = {"features": cols, "sigma": float(ganador["RMSE"]), "ganador": ganador["Modelo"],
        "rmse": float(ganador["RMSE"]), "mae": float(ganador["MAE"]), "r2": float(ganador["R2"]),
        "rmse_persistencia": float(rm_p), "rmse_media": float(rm_m),
        "mejora_vs_persistencia_pct": float(100 * (rm_p - ganador["RMSE"]) / rm_p)}
    TABLAS[v] = tabla
for v, t in TABLAS.items():
    print(f"\n--- {v} ---"); print(t.round(4).to_string(index=False))


===== temperatura_c =====


  filas entrenamiento=245273  prueba=61433
  Referencia: repetir el ultimo valor: RMSE=0.2365  MAE=0.1769  R2=0.9935
  Referencia: media de entrenamiento: RMSE=8.1750  MAE=7.6283  R2=-6.7360


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


  LinearRegression: RMSE=0.1299  MAE=0.0957  R2=0.9980


  RandomForest: RMSE=1.7089  MAE=1.1656  R2=0.6620



===== oxigeno_mgl =====


  filas entrenamiento=245273  prueba=61433
  Referencia: repetir el ultimo valor: RMSE=0.2096  MAE=0.1221  R2=0.9757
  Referencia: media de entrenamiento: RMSE=2.1246  MAE=1.8296  R2=-1.4966


  LinearRegression: RMSE=0.1422  MAE=0.0964  R2=0.9888


  RandomForest: RMSE=0.3928  MAE=0.2064  R2=0.9147



--- temperatura_c ---
                             Modelo   RMSE    MAE      R2
Referencia: repetir el ultimo valor 0.2365 0.1769  0.9935
 Referencia: media de entrenamiento 8.1750 7.6283 -6.7360
                   LinearRegression 0.1299 0.0957  0.9980
                       RandomForest 1.7089 1.1656  0.6620

--- oxigeno_mgl ---
                             Modelo   RMSE    MAE      R2
Referencia: repetir el ultimo valor 0.2096 0.1221  0.9757
 Referencia: media de entrenamiento 2.1246 1.8296 -1.4966
                   LinearRegression 0.1422 0.0964  0.9888
                       RandomForest 0.3928 0.2064  0.9147


## 5. Lectura honesta del resultado

Predecir una hora adelante en agua es dificil de mejorar: "repetir el ultimo valor" suele ser una referencia fuerte. Si el modelo no la supera, se dice asi.

In [6]:
for v, m in META["variables"].items():
    mej = m["mejora_vs_persistencia_pct"]
    if mej <= 0:
        lectura = "NO supera a repetir el ultimo valor."
    elif mej < 5:
        lectura = "mejora marginal (<5 %) frente a repetir el ultimo valor."
    else:
        lectura = "mejora material frente a repetir el ultimo valor."
    print(f"{v}: ganador={m['ganador']}  RMSE={m['rmse']:.4f}  vs persistencia={m['rmse_persistencia']:.4f}  ({mej:+.1f} %)  -> {lectura}")

temperatura_c: ganador=LinearRegression  RMSE=0.1299  vs persistencia=0.2365  (+45.1 %)  -> mejora material frente a repetir el ultimo valor.
oxigeno_mgl: ganador=LinearRegression  RMSE=0.1422  vs persistencia=0.2096  (+32.1 %)  -> mejora material frente a repetir el ultimo valor.


## 6. Umbrales de advertencia por estacion

Cada estacion tiene su propia historia: la advertencia se define con el percentil 5 y 95 de lo observado en la parte de ENTRENAMIENTO. La norma (critico) se declara aparte en el cuaderno C.

In [7]:
umb = (base.where(F.col("ts5") <= CORTE)
       .groupBy("estacion_id", "variable")
       .agg(F.percentile_approx("valor", 0.05).alias("p05"), F.percentile_approx("valor", 0.95).alias("p95"),
            F.count("*").alias("n")))
umb.orderBy("estacion_id", "variable").show(40, truncate=False)
META["umbrales"] = {f"{r['estacion_id']}|{r['variable']}": {"p05": r["p05"], "p95": r["p95"], "n": r["n"]} for r in umb.collect()}

+---------------+-------------+----+----+-----+
|estacion_id    |variable     |p05 |p95 |n    |
+---------------+-------------+----+----+-----+
|USGS-01491000  |oxigeno_mgl  |4.7 |8.3 |20703|
|USGS-01491000  |temperatura_c|16.3|28.0|20703|
|USGS-01579550  |oxigeno_mgl  |5.9 |10.7|20678|
|USGS-01579550  |temperatura_c|20.2|29.8|20678|
|USGS-01589290  |temperatura_c|15.3|25.4|20435|
|USGS-01594441  |oxigeno_mgl  |6.2 |8.7 |20662|
|USGS-01594441  |temperatura_c|17.3|27.0|20662|
|USGS-01640152  |oxigeno_mgl  |6.5 |9.6 |20676|
|USGS-01640152  |temperatura_c|14.8|26.1|20676|
|USGS-0164578734|oxigeno_mgl  |6.5 |9.0 |20696|
|USGS-0164578734|temperatura_c|15.5|24.8|20699|
|USGS-0164579522|oxigeno_mgl  |4.1 |7.9 |20458|
|USGS-0164579522|temperatura_c|15.2|24.0|20657|
|USGS-01646500  |oxigeno_mgl  |6.8 |10.0|20663|
|USGS-01646500  |temperatura_c|18.8|29.7|20692|
|USGS-01649190  |oxigeno_mgl  |7.0 |9.4 |20464|
|USGS-01649190  |temperatura_c|15.6|25.2|20669|
|USGS-01649500  |oxigeno_mgl  |5.7 |10.8

In [8]:
with open(f"{RUTA_MODELOS}/meta.json", "w", encoding="utf-8") as f:
    json.dump(META, f, ensure_ascii=False, indent=1)
print("Modelos y metadatos guardados en:", RUTA_MODELOS)
print(os.listdir(RUTA_MODELOS))

Modelos y metadatos guardados en: /opt/s12-u2-usgs/artifacts/modelos_patricia
['meta.json', 'modelo_oxigeno_mgl_1h', 'modelo_temperatura_c_1h']
